# SWAP-Stress: SMAP L3 soil moisture

**SMAP Enhanced L3** (SPL3SMP_E) is the pipeline's one dynamic input. The SMAP
radiometer measures L-band (1.4 GHz) brightness temperature, which is sensitive
to moisture in the top ~5 cm of soil. We use the 9 km Enhanced product on
EASE-Grid 2.0, converted to daily GeoTIFFs by
`swapstress.features.smap_download`.

1. The SMAP product family, and why L3 rather than L4
2. The grid the whole pipeline is aligned to
3. A single day
4. Temporal coverage
5. Multi-year summer composites
6. Retrieval gaps, and what stage 06 does about them

In [ ]:
from __future__ import annotations

import os

import matplotlib.pyplot as plt
import numpy as np
import rasterio

from swapstress.config import load_config

# ---------------------------------------------------------------------------
# Paths come from the run configs the pipeline itself uses, so this notebook and
# stage 05 always read the same rasters. REPO is the only thing you may need to
# change -- point it at your checkout if the notebook is run from elsewhere.
# ---------------------------------------------------------------------------
REPO = os.path.abspath(os.environ.get("SWAPSTRESS_REPO", "."))

predict_cfg = load_config(
    os.path.join(REPO, "configs", "predict_9km_global_pruned.toml"), {}
)

SMAP_DIR = predict_cfg["smap_dir"]
SMAP_GF_DIR = SMAP_DIR + "_gapfilled"

OUT_DIR = os.path.join("notebooks", "_outputs")
os.makedirs(OUT_DIR, exist_ok=True)

print("SMAP_DIR:   ", SMAP_DIR)
print("SMAP_GF_DIR:", SMAP_GF_DIR)

## 1) The SMAP product family

| Product | Level | Resolution | Temporal | Description |
|---------|-------|-----------|----------|-------------|
| SPL1CTB | L1C | 36 km | Half-orbit | Calibrated brightness temperature |
| SPL2SMP | L2 | 36 km | Half-orbit | Soil moisture from Tb, single-channel algorithm |
| SPL2SMP_E | L2 | 9 km | Half-orbit | Enhanced 9 km via Backus-Gilbert interpolation of Tb |
| **SPL3SMP_E** | **L3** | **9 km** | **Daily** | **Daily composite of L2 Enhanced — what we use** |
| SPL3FTP | L3 | 9 km | Daily | Freeze/thaw state |
| SPL4SMGP | L4 | 9 km | 3-hourly | Assimilated soil moisture from the NASA Catchment LSM |
| SPL4SMAU | L4 | 9 km | 3-hourly | Analysis update (observation minus forecast) |

### Why not L4

L4 assimilates brightness temperature into NASA's Catchment Land Surface Model,
which carries its own pedotransfer-derived soil hydraulic parameters internally.
Feeding it to a model whose whole purpose is to learn θ → ψ would be leakage: the
model would learn to invert L4's built-in hydraulic assumptions rather than the
real relationship. **L4 is never a feature.** The only SMAP inputs the pipeline
permits are the L3 vegetation water content climatology as a static covariate and
L3 `soil_moisture` as the daily θ at inference time.

## 2) The grid

`swapstress.features.smap_download` owns the EASE-Grid 2.0 definition — the
global M09 grid and the CONUS window cut from it. Every covariate raster is
reprojected onto exactly this grid by
`swapstress.features.reproject_to_ease2`, which reads the same constants, so the
static stack and the daily soil moisture line up pixel for pixel with no
resampling at prediction time.

In [ ]:
from swapstress.features.smap_download import (
    EASE2_CRS,
    GLOBAL_COLS,
    GLOBAL_ROWS,
    MAP_SCALE,
    resolve_ease2_grid,
)

print(f"EASE-Grid 2.0 global M09: {GLOBAL_COLS} x {GLOBAL_ROWS} px @ {MAP_SCALE:.3f} m")
print(f"CRS: {EASE2_CRS}")

for scope in ("global", "conus"):
    rows, cols, transform = resolve_ease2_grid(scope)
    print(f"\n{scope}: {cols.stop - cols.start} x {rows.stop - rows.start} px")
    print(f"  rows {rows.start}:{rows.stop}  cols {cols.start}:{cols.stop}")
    print(f"  transform {transform}")

## 3) A single day

`swapstress.inference.gapfill.discover_source_rasters` is the shared date-keyed
raster index — the same function stage 06 uses to find its inputs — so the days
listed here are the days the pipeline sees.

In [ ]:
from datetime import date

from swapstress.inference.gapfill import discover_source_rasters

smap_files = discover_source_rasters(SMAP_DIR, prefix="smap_sm")
print(f"{len(smap_files):,} daily SMAP rasters, {min(smap_files)} to {max(smap_files)}")

REF_DATE = date(2020, 7, 25)
ref_file = smap_files[REF_DATE]

with rasterio.open(ref_file) as src:
    print(f"\nFile:        {os.path.basename(ref_file)}")
    print(f"Shape:       {src.width} x {src.height} (cols x rows)")
    print(f"CRS:         {src.crs}")
    print(f"Pixel size:  {src.transform.a:.3f} m")
    print(f"Band:        {src.descriptions[0]}")
    data = src.read(1)
    extent = [src.bounds.left, src.bounds.right, src.bounds.bottom, src.bounds.top]

valid = data[~np.isnan(data)]
print(
    f"\nValid pixels: {len(valid):,} / {data.size:,} "
    f"({100 * len(valid) / data.size:.1f}%)"
)
print(
    f"Range: {valid.min():.4f} - {valid.max():.4f} m3/m3, "
    f"mean {valid.mean():.4f}, median {np.median(valid):.4f}"
)

In [ ]:
from swapstress.figures import basemap

states = basemap.load_conus_states(crs=EASE2_CRS.to_epsg())

fig, ax = plt.subplots(figsize=(12, 6), dpi=120)
im = ax.imshow(
    data, extent=extent, origin="upper", cmap="viridis", vmin=0.02, vmax=0.55
)
states.boundary.plot(ax=ax, color="0.3", linewidth=0.4)
ax.set_xlim(extent[0], extent[1])
ax.set_ylim(extent[2], extent[3])

fig.colorbar(im, ax=ax, shrink=0.7, pad=0.02, label="Volumetric water content (m3/m3)")
ax.set_title(f"SMAP L3 Enhanced - {REF_DATE}", fontsize=13)
ax.set_xlabel("Easting (m, EPSG:6933)")
ax.set_ylabel("Northing (m, EPSG:6933)")

fig.tight_layout()
out = os.path.join(OUT_DIR, "smap_l3_single_day.png")
fig.savefig(out, dpi=200)
plt.show()
print("Saved:", os.path.abspath(out))

## 4) Temporal coverage

Days present per month, straight off the date index. Gaps are radiometer
downtime and the occasional missing granule; SMAP began operating in April 2015.

In [ ]:
import pandas as pd

dates = pd.DatetimeIndex(sorted(smap_files))
counts = (
    pd.Series(1, index=dates)
    .groupby([dates.year, dates.month])
    .size()
    .unstack(fill_value=0)
    .reindex(columns=range(1, 13), fill_value=0)
)

fig, ax = plt.subplots(figsize=(10, 5), dpi=120)
im = ax.imshow(counts.to_numpy(), aspect="auto", cmap="YlGnBu", vmin=0, vmax=31)

ax.set_yticks(range(len(counts.index)))
ax.set_yticklabels(counts.index, fontsize=8)
ax.set_xticks(range(12))
ax.set_xticklabels(list("JFMAMJJASOND"))

for i in range(counts.shape[0]):
    for j in range(12):
        n = counts.iat[i, j]
        if n:
            ax.text(j, i, str(n), ha="center", va="center", fontsize=6)

fig.colorbar(im, ax=ax, shrink=0.8, pad=0.02, label="Days with data")
ax.set_title(f"SMAP L3 Enhanced temporal coverage ({len(smap_files):,} daily rasters)")
ax.set_xlabel("Month")
ax.set_ylabel("Year")

fig.tight_layout()
out = os.path.join(OUT_DIR, "smap_l3_temporal_coverage.png")
fig.savefig(out, dpi=200)
plt.show()
print("Saved:", os.path.abspath(out))

## 5) Summer composites

Mean and standard deviation over summer days (DOY 150-250) across several years.
The mean is the arid-West / humid-East gradient; the standard deviation is where
soil moisture actually varies, which is where a dynamic θ input earns its keep.

In [ ]:
summer = [
    d
    for d in sorted(smap_files)
    if 2018 <= d.year <= 2023 and 150 <= d.timetuple().tm_yday <= 250
]
sample = summer[:: max(1, len(summer) // 100)][:100]
print(f"Summer days available: {len(summer):,}; stacking {len(sample)}")

stack = np.full((len(sample), *data.shape), np.nan, dtype=np.float32)
for i, d in enumerate(sample):
    with rasterio.open(smap_files[d]) as src:
        stack[i] = src.read(1)

obs_count = np.sum(~np.isnan(stack), axis=0)
mean_sm = np.where(obs_count >= 10, np.nanmean(stack, axis=0), np.nan)
std_sm = np.where(obs_count >= 10, np.nanstd(stack, axis=0), np.nan)

print(f"Mean VWC: {np.nanmin(mean_sm):.4f} - {np.nanmax(mean_sm):.4f}")
print(f"Std VWC:  {np.nanmin(std_sm):.4f} - {np.nanmax(std_sm):.4f}")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5), dpi=120)

for ax, arr, title, cmap, vmin, vmax, label in [
    (
        ax1,
        mean_sm,
        "Mean soil moisture, summer 2018-2023",
        "viridis",
        0.02,
        0.45,
        "VWC (m3/m3)",
    ),
    (
        ax2,
        std_sm,
        "Std dev of soil moisture, summer 2018-2023",
        "magma",
        0,
        0.10,
        "Std dev (m3/m3)",
    ),
]:
    im = ax.imshow(arr, extent=extent, origin="upper", cmap=cmap, vmin=vmin, vmax=vmax)
    states.boundary.plot(ax=ax, color="0.5", linewidth=0.4)
    ax.set_xlim(extent[0], extent[1])
    ax.set_ylim(extent[2], extent[3])
    fig.colorbar(im, ax=ax, shrink=0.7, pad=0.02, label=label)
    ax.set_title(title)
    ax.set_xlabel("Easting (m)")
    ax.set_ylabel("Northing (m)")

fig.tight_layout()
out = os.path.join(OUT_DIR, "smap_l3_summer_composite.png")
fig.savefig(out, dpi=200)
plt.show()
print("Saved:", os.path.abspath(out))

## 6) Retrieval gaps

A single SMAP pass covers roughly a third of the grid on any given day. The
released product handles that at the **prediction** end, not the input end:
stage 05 predicts only where there is a retrieval (Level 1), and stage 06
(`swapstress-gapfill`) interpolates the resulting rasters along the time axis to
give Level 2.

```bash
uv run swapstress-gapfill --config configs/gapfill_9km_global_pruned.toml
```

`swapstress.inference.gapfill.interpolate_pixel` is the per-pixel operation, and
`run_gapfill` is the driver. The comparison below only runs if you have also
gap-filled the soil moisture rasters themselves into `SMAP_GF_DIR`; that is a
diagnostic, not part of the release chain.

In [ ]:
if not os.path.isdir(SMAP_GF_DIR):
    print(f"No gap-filled SMAP directory at {SMAP_GF_DIR}; skipping the comparison.")
    print("The released chain gap-fills the prediction rasters (stage 06), not the")
    print("soil moisture input, so this directory is optional.")
else:
    gf_files = discover_source_rasters(SMAP_GF_DIR, prefix="smap_sm")
    print(f"Raw:        {len(smap_files):,} rasters")
    print(f"Gap-filled: {len(gf_files):,} rasters")

    def valid_per_day(files):
        counts = {}
        for d, path in sorted(files.items()):
            with rasterio.open(path) as src:
                counts[d] = int(np.sum(~np.isnan(src.read(1))))
        return counts

    raw_counts = valid_per_day(smap_files)
    gf_counts = valid_per_day(gf_files)

    print(f"\nRaw  median valid px/day: {np.median(list(raw_counts.values())):,.0f}")
    print(f"GF   median valid px/day: {np.median(list(gf_counts.values())):,.0f}")

    fig, ax = plt.subplots(figsize=(14, 4), dpi=120)
    ax.bar(
        list(raw_counts),
        list(raw_counts.values()),
        width=1,
        color="steelblue",
        alpha=0.8,
        label="Raw",
    )
    ax.bar(
        list(gf_counts),
        list(gf_counts.values()),
        width=1,
        color="tomato",
        alpha=0.4,
        label="Gap-filled",
    )
    ax.set_xlabel("Date")
    ax.set_ylabel("Valid pixels")
    ax.set_title("SMAP L3 valid-pixel count per day")
    ax.legend()
    fig.tight_layout()
    plt.show()

## Next

`06_9k_feature_data.ipynb` tours the static covariate rasters that share this
grid, and `07_inference.ipynb` runs the model over both.